## Setup
Import libraries and set plot defaults.

In [ ]:
# To install the required packages, run:
# !pip install numpy pandas scipy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import ttest_ind, pearsonr, multivariate_normal
import os, warnings

warnings.filterwarnings("ignore")
os.makedirs("figures", exist_ok=True)

# Publication-quality plot defaults
plt.rcParams.update({
    "font.size": 12, "axes.labelsize": 13, "axes.titlesize": 14,
    "lines.linewidth": 2.0, "figure.figsize": (10, 6),
    "savefig.dpi": 150, "savefig.bbox": "tight",
})
print("Libraries imported")

## Global settings
Set experiment constants

In [ ]:
N_PARTICIPANTS = 50
N_TRIALS       = 160
BLOCK_SIZE     = 40
V0             = 0.5                          # initial value for both stimuli
PROB_A         = [0.60, 0.80, 0.60, 0.65]    # P(sound | chose A) per block

print(f"Design: {N_PARTICIPANTS} participants × {N_TRIALS} trials, {len(PROB_A)} blocks of {BLOCK_SIZE}")

## Data Exploration

### Load data

In [ ]:
# Data path

DATA_DIR = r"C:\Users\xiaon\OneDrive\申请\code\ccn assignment 2\data" ## adjust this path to where the data files are
#   Windows:  r"C:\Users\yourname\Desktop\data"
#   Mac/Linux: "/home/yourname/data"
#   Same folder as notebook: "data"

# Load data
raw_choices = pd.read_csv(os.path.join(DATA_DIR, "inst_choices.csv"),  header=None).values
outcomes    = pd.read_csv(os.path.join(DATA_DIR, "inst_outcomes.csv"), header=None).values
stai        = pd.read_csv(os.path.join(DATA_DIR, "stai_scores.csv"),  header=None).values.flatten()

# Print data
print(f"raw_choices shape: {raw_choices.shape}   unique values: {np.unique(raw_choices)}")
print(f"outcomes shape:    {outcomes.shape}   unique values: {np.unique(outcomes)}")
print(f"stai shape:        {stai.shape}")

### Recode choices

Recode 1 = A, 2 = B in the raw file into 1 = A, 0 = B.

In [ ]:
choices = (raw_choices == 1).astype(int)

# Print data to check
print(f"Before (subj 1, first 10): {raw_choices[0, :10]}")
print(f"After  (subj 1, first 10): {choices[0, :10]}")
print(f"Subject 1 chose A: {choices[0].sum()}/{N_TRIALS} = {choices[0].mean()*100:.1f}% (hint: ~18%)")

### Descriptive statistics

The STAI form Y2 measures trait anxiety.

In [ ]:
print("STAI-Y2 (Trait Anxiety)")
print(f"  Mean:   {stai.mean():.2f}")
print(f"  SD:     {stai.std():.2f}")
print(f"  Median: {np.median(stai):.2f}")
print(f"  Range:  {stai.min()} – {stai.max()}")

### Clinical cutoff for group classification

Sanity check on whether the participants' actual anxiety scores match how they were recruited.

**Self-identification at recruitment:**
- Indices 0–24 (participants 1–25): recruited as anxious
- Indices 25–49 (participants 26–50): recruited as calm

**STAI-Y2 questionnaire score**. Clinical cutoff is 43:
- Score ≤ 43: healthy control
- Score > 43: clinically anxious

> **Note:** Regardless of any mismatch, we use the original recruitment groups (0–24 = high, 25–49 = low) for all subsequent analyses.

In [ ]:
CUTOFF = 43
healthy_idx = np.where(stai <= CUTOFF)[0]

print(f"Healthy controls (STAI ≤ {CUTOFF}): n = {len(healthy_idx)}")
print(f"Indices: {healthy_idx}")

recruited_anxious = set(range(0, 25))
recruited_calm    = set(range(25, 50))
healthy_set       = set(healthy_idx)

print(f"\nRecruited anxious but scored healthy: {sorted(recruited_anxious & healthy_set)}")
print(f"Recruited calm but scored anxious:    {sorted(recruited_calm - healthy_set)}")
print(f"\nFor all analyses below: indices 0–24 = HIGH anxiety, 25–49 = LOW anxiety.")

### Assess task performance

Check whether participants learned to avoid the aversive stimulus, or were simply choosing randomly.

Baseline (random responding): If a participant chose A or B with equal probability (50/50) on every trial, they would expect to hear 80 aversive sounds out of 160 trials.

- Mean actual sounds < 80 means participants successfully learned to avoid the worse stimulus
- Mean actual sounds ≈ 80 means random-level performance

In [ ]:
pct_a = choices.sum(axis=1) / N_TRIALS * 100
print(f"Subject 1 chose A: {pct_a[0]:.1f}% (hint ~18%)") # Verify code
print(f"Mean % choosing A:  {pct_a.mean():.2f}%") # Report how often each participant chose stimulus A. Since A carries higher aversive probability in every block, effective learners should choose A infrequently

expected_random = sum(BLOCK_SIZE * 0.5 for _ in PROB_A)   # = 80
actual_sounds   = outcomes.sum(axis=1)

print(f"\nExpected sounds (random): {expected_random:.0f}")
print(f"Mean actual sounds:       {actual_sounds.mean():.2f} ± {actual_sounds.std():.2f}")
print(f"Range:                    {actual_sounds.min()} – {actual_sounds.max()}")
diff = expected_random - actual_sounds.mean()
print(f"\nParticipants heard {diff:.1f} fewer sounds than chance")

### Summary data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["tab:red"] * 25 + ["tab:blue"] * 25

axes[0].bar(range(N_PARTICIPANTS), stai, color=colors, alpha=0.7)
axes[0].axhline(CUTOFF, color="black", ls="--", lw=1.5, label=f"Cutoff = {CUTOFF}")
axes[0].axvline(24.5, color="grey", ls=":", lw=1.5)
axes[0].set_xlabel("Participant"); axes[0].set_ylabel("STAI-Y2 score")
axes[0].set_title("(a) Trait Anxiety"); axes[0].legend()

axes[1].bar(range(N_PARTICIPANTS), actual_sounds, color=colors, alpha=0.7)
axes[1].axhline(expected_random, color="black", ls="--", lw=1.5, label="Chance = 80")
axes[1].axvline(24.5, color="grey", ls=":", lw=1.5)
axes[1].set_xlabel("Participant"); axes[1].set_ylabel("Aversive sounds")
axes[1].set_title("(b) Task Performance"); axes[1].legend()

for ax in axes:
    yl = ax.get_ylim()
    ax.text(12, yl[1]*0.93, "High anx.", ha="center", color="tab:red", fontweight="bold")
    ax.text(37, yl[1]*0.93, "Low anx.",  ha="center", color="tab:blue", fontweight="bold")

plt.tight_layout(); plt.savefig("figures/task_a.png"); plt.show()